In [ ]:
import os
from tqdm.auto import tqdm
import matplotlib.pyplot as plt 

import numpy as np 
from PIL import Image
import torch
import torch.nn as nn 
from torch.utils.data import DataLoader, Dataset
import torch.nn.functional as F 

import albumentations as A
from albumentations.pytorch import ToTensorV2

from torchinfo import summary
from sklearn.model_selection import train_test_split

In [ ]:
seed = 42
batch_size = 64

In [ ]:
def set_seed(seed=42):
    # np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

In [ ]:
data_path = '../data/raw/dataset_CIDIS_sisr_x8/thermal'
print(data_path)

In [ ]:
train_path_LR = os.path.join(data_path, 'train', 'LR_x8')
train_path_GT = os.path.join(data_path, 'train', 'GT')
print(train_path_LR)
print(train_path_GT)

In [ ]:
val_path_LR = os.path.join(data_path, 'val', 'LR_x8')
val_path_GT = os.path.join(data_path, 'val', 'GT')
print(val_path_LR)
print(val_path_GT)

In [ ]:
test_path = os.path.join(data_path, 'test', 'sisr_x8', 'LR_x8')
print(test_path)

In [ ]:
class CIDISDataset(Dataset):
    def __init__(self, data_path, transforms=None, split='train'):
        self.split = split
        self.transforms = transforms
        if split != 'test':
            self.LR_dir = os.path.join(data_path, split, 'LR_x8')
            self.GT_dir = os.path.join(data_path, split, 'GT')

            self.LR_images = sorted(os.listdir(self.LR_dir))
            self.GT_images = sorted(os.listdir(self.GT_dir))
        else: 
            self.LR_dir = os.path.join(data_path, split, 'sisr_x8', 'LR_x8')
            self.LR_images = os.listdir(self.LR_dir)

    def __len__(self):
        return len(self.LR_images)

    def __getitem__(self, idx):

        LR_image_path = os.path.join(self.LR_dir, self.LR_images[idx])
        LR_image = Image.open(LR_image_path).convert('RGB')
        LR_image = np.array(LR_image)
        if self.split == 'test':
            return LR_image
        
        GT_image_path = os.path.join(self.GT_dir, self.GT_images[idx])
        GT_image = Image.open(GT_image_path).convert('RGB')
        GT_image = np.array(GT_image)

        if self.transforms:
            transform_image = self.transforms(image=LR_image, GT_image=GT_image)
            LR_image = transform_image['image']
            GT_image = transform_image['GT_image']

        return LR_image, GT_image    


In [ ]:
train_transforms = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    ToTensorV2()
], additional_targets={'GT_image': 'image'})

val_test_transforms = A.Compose([
    ToTensorV2()
], additional_targets={'GT_image': 'image'})

In [ ]:
train_dataset = CIDISDataset(data_path, transforms=train_transforms, split='train')
val_dataset = CIDISDataset(data_path, transforms=val_test_transforms, split='val')
test_dataset = CIDISDataset(data_path, transforms=val_test_transforms, split='test')

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

## pipeline model 1: EDSR



In [ ]:
class ChannelAttention(nn.Module):
    def __init__(self, num_features, reduction=16):
        self.avg = nn.AdaptiveAvgPool2d(1)
        self.conv = nn.Sequential(
            nn.Conv2d(num_features, num_features // reduction, kernel_size=1, padding=0, bias=True),
            nn.ReLU(inplace=True),
            nn.Conv2d(num_features // reduction, num_features, kernel_size=1, padding=0, bias=True),
            nn.Sigmoid()
        )

    def forward(self, x):
        y = self.avg(x) 
        y = self.conv(y)
        return x * y
        

In [ ]:
class ResBlock(nn.Module):
    def __init__(self, num_features, res_scale=0.1):
        super().__init__()
        self.conv1 = nn.Conv2d(num_features, num_features, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(num_features, num_features, kernel_size=3, padding=1)
        self.ChannelAttention = ChannelAttention(num_features, reduction=16)
        self.relu = nn.ReLU()
        self.res_scale = res_scale
    
    def forward(self, x):
        residual = x
        out = self.relu(self.conv1(x))
        out = self.conv2(x)
        out = self.ChannelAttention(out) + self.res_scale + residual
        return out

In [ ]:
class Upsampler(nn.Module):
    """Upsample for x8 SR"""
    def __init__(self, in_channels, scale_factor=8):
        super().__init__()
        self.upsampler = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(in_channels, in_channels * 4, kernel_size=3, padding=1),
                nn.PixelShuffle(2),
                nn.ReLU(inplace=True)
            ) for _ in range(math.log2(scale_factor))
        ])
    
    def forward(self, x):
        for block in self.upsampler:
            x = block(x)
        return x

In [ ]:
class EDSR(nn.Module):
    def __init__(self, in_channels, out_channels, num_features, num_blocks, res_scale=0.1):
        super().__init__()

        out_channels = out_channels if out_channels is not None else in_channels

        # Head
        self.head = nn.Conv2d(in_channels, num_features, kernel_size=3, padding=1)

        # Body (Stack ResBlock)
        self.StackResBlock = nn.ModuleList([
            ResBlock(num_features, res_scale=res_scale) for _ in range(num_blocks)
        ])

        # Upsampler 
        self.upsampler = Upsampler(num_features, scale_factor=8)

        # Additional upsampling conv for fine adjustment
        self.upsample_conv = nn.Conv2d(num_features, num_features, kernel_size=3, padding=1)

        # Tail
        self.tail = nn.Conv2d(num_features, out_channels, kernel_size=3, padding=1)

    def forward(self, x):
        x = self.head(x)
        residual = x
        for block in self.StackResBlock:
            x = block(x)
        x = x + residual
        x = self.upsampler(x)
        x = self.upsample_conv(x)
        x = self.tail(x)
        return x